In [751]:
from google.colab import drive
import os
import pandas as pd
from scipy import stats
import numpy as np

In [752]:

drive.mount('/content/drive')

DATA_DIR = '/content/drive/My Drive/data'

# 3. Automatically change the working directory so relative paths work
if os.path.exists(DATA_DIR):
    os.chdir(DATA_DIR)
    print(f"Successfully connected! Current working directory: {os.getcwd()}")
else:
    print(f"Error: The folder '{DATA_DIR}' could not be found.")
    print("Make sure to add a shortcut of the 'data' folder to your 'My Drive'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully connected! Current working directory: /content/drive/.shortcut-targets-by-id/1_ib6SYLLQ6Asjybns0GSlgBDwFQrnmkj/data


### Uploading Data and Addressing Missing Values and Outliers

## Data Cleaning Methodology

The datasets are cleaned by addressing missing values and potential outliers.

### Missing values

Missing values are handled according to the meaning of the variable rather than
using one imputation method for every column.

The following rules are used:

- Sentinel values representing missing data are converted to `NaN`.
- Rows missing an essential measurement are removed when the observation cannot
  be used for the analysis.
- Short gaps in sequential environmental measurements may be filled using
  linear interpolation when nearby observations provide a reasonable estimate.
- Missing categorical metadata is represented as `"Unknown"` when appropriate.
- Measurement or metadata values that cannot be reliably inferred are left as
  `NaN`.
- Median imputation is used only when a numerical variable can reasonably be
  represented by a typical value and interpolation is not appropriate.

### Outliers

The Interquartile Range (IQR) method is used consistently for outlier screening.

For each measurement:

- IQR = Q3 - Q1
- Lower bound = Q1 - 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Values outside these bounds are considered statistical outliers. However,
statistical outliers are not automatically removed. They are investigated using
their units, sampling context, location, quality information, and physical
plausibility.

Values are removed only when there is evidence that they are clearly erroneous
or physically implausible.

In [753]:
# Load all datasets
bradentonFL = pd.read_csv("bradentonFL_tempData.csv")
floridaCyanotoxins = pd.read_csv("florida_cyanotoxins_2015-01-01_to_2026-06-17.csv")
habsos = pd.read_csv("habsos_cellcounts.csv")
motemarine = pd.read_csv("MoteMarine_BottomTemp.csv")
nutrientsFL = pd.read_csv("NutrientsFL_2006_2025.csv")
sarasota = pd.read_csv("Sarasota_Wind&Temp.csv")

/tmp/ipykernel_1220/3135485723.py:3: DtypeWarning: Columns (48) have mixed types. Specify dtype option on import or set low_memory=False.
  floridaCyanotoxins = pd.read_csv("florida_cyanotoxins_2015-01-01_to_2026-06-17.csv")
/tmp/ipykernel_1220/3135485723.py:4: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  habsos = pd.read_csv("habsos_cellcounts.csv")
/tmp/ipykernel_1220/3135485723.py:7: DtypeWarning: Columns (1,2,4,6,8,10,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  sarasota = pd.read_csv("Sarasota_Wind&Temp.csv")


## bradentonFL_tempData.csv

In [754]:
print("Shape:", bradentonFL.shape)

Shape: (4018, 8)


In [755]:
bradentonFL.head()

,COOPID,YEAR,MONTH,DAY,PRECIPITATION,MAX TEMP,MIN TEMP,MEAN TEMP
0,80945,2015,1,1,0.0,74.0,60.0,
1,80945,2015,1,2,0.0,78.0,64.0,
2,80945,2015,1,3,0.0,85.0,70.0,
3,80945,2015,1,4,0.0,83.0,69.0,
4,80945,2015,1,5,0.0,75.0,61.0,


In [756]:
# Check missing values before cleaning.
print("Missing values:")
print(bradentonFL.isna().sum())

Missing values:
COOPID            0
 YEAR             0
 MONTH            0
 DAY              0
 PRECIPITATION    0
 MAX TEMP         0
 MIN TEMP         0
 MEAN TEMP        0
dtype: int64


In [757]:
print("\nColumn names:")
print(bradentonFL.columns)


Column names:
Index(['COOPID', ' YEAR', ' MONTH', ' DAY', ' PRECIPITATION', ' MAX TEMP',
       ' MIN TEMP', ' MEAN TEMP'],
      dtype='object')


In [758]:
# Rename columns for easier reference
bradentonFL.rename(
    columns={
        'COOPID': "coop_id",
        ' YEAR': 'year',
        ' MONTH': 'month',
        ' DAY': 'day',
        ' PRECIPITATION': 'precipitation',
        ' MAX TEMP': 'max_temp',
        ' MIN TEMP': 'min_temp',
        ' MEAN TEMP': 'mean_temp'
    },
    inplace=True
)

In [759]:
# Remove mean temperature because it will not needed for the analysis
bradentonFL.drop(columns='mean_temp', inplace=True)

In [760]:
bradentonFL.describe()

,coop_id,year,month,day,precipitation,max_temp,min_temp
count,4018.0,4018.00000,4018.000000,4018.000000,4018.000000,4018.000000,4018.000000
mean,80945.0,2020.00000,6.522648,15.730463,0.028726,84.121379,66.079841
std,0.0,3.16275,3.449210,8.801536,3.565303,11.761406,12.512208
min,80945.0,2015.00000,1.000000,1.000000,-99.990000,-99.900000,-99.900000
25%,80945.0,2017.00000,4.000000,8.000000,0.000000,80.000000,60.000000
50%,80945.0,2020.00000,7.000000,16.000000,0.000000,86.000000,69.000000
75%,80945.0,2023.00000,10.000000,23.000000,0.040000,91.000000,75.000000
max,80945.0,2025.00000,12.000000,31.000000,11.650000,101.000000,83.000000


### Bradenton Missing Values

The precipitation and temperature columns contain sentinel values such as
`-99.99` and `-99.9`. These values represent missing observations rather than
actual environmental measurements, so they are converted to `NaN`.

Missing weather measurements are not median-imputed because doing so would
create artificial weather observations for days when measurements were not
recorded.

In [761]:
# Check for sentinel values representing missing temperature or precipitation data
bradentonFL[
    ["precipitation", "max_temp", "min_temp"]
].isin([-99.99, -99.9]).sum()

,0
precipitation,5
max_temp,7
min_temp,8


In [762]:
# Replace sentinel values with NaN so they are treated as missing values
bradentonFL["precipitation"] = bradentonFL["precipitation"].replace([-99.99, -99.9], np.nan)
bradentonFL["max_temp"] = bradentonFL["max_temp"].replace(-99.9, np.nan)
bradentonFL["min_temp"] = bradentonFL["min_temp"].replace(-99.9, np.nan)

In [763]:
# Check for any remaining negative precipitation values
print(
    bradentonFL[
        ["precipitation", "max_temp", "min_temp"]
    ].isin([-99.99, -99.9]).sum()
)

print("\nRemaining missing values:")
print(bradentonFL.isna().sum())

precipitation    0
max_temp         0
min_temp         0
dtype: int64

Remaining missing values:
coop_id          0
year             0
month            0
day              0
precipitation    5
max_temp         7
min_temp         8
dtype: int64


In [765]:
# Check for any remaining negative precipitation values.
bradentonFL[
    bradentonFL["precipitation"] < 0
][
    ["year", "month", "day", "precipitation"]
]

,year,month,day,precipitation


In [767]:
# Check whether minimum temperature exceeds maximum temperature.
print(
    "Rows where min_temp > max_temp:",
    (bradentonFL["min_temp"] > bradentonFL["max_temp"]).sum()
)

Rows where min_temp > max_temp: 0


### Bradenton Outlier Detection

IQR is applied to precipitation, maximum temperature, and minimum temperature.
The date variables and station identifier are not treated as environmental
measurement outliers.

IQR outliers are investigated rather than automatically deleted because extreme
weather conditions can be legitimate observations.

In [768]:
# Variables used for IQR outlier screening.
measurement_cols = [
    "precipitation",
    "max_temp",
    "min_temp"
]

# Calculate Q1, Q3, and IQR.
Q1 = bradentonFL[measurement_cols].quantile(0.25)
Q3 = bradentonFL[measurement_cols].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bounds:")
print(lower_bound)

print("\nUpper bounds:")
print(upper_bound)

# Identify IQR outliers.
outliers = (
    (bradentonFL[measurement_cols] < lower_bound) |
    (bradentonFL[measurement_cols] > upper_bound)
)

print("\nNumber of IQR outliers:")
print(outliers.sum())

Lower bounds:
precipitation    -0.06
max_temp         63.50
min_temp         37.50
dtype: float64

Upper bounds:
precipitation      0.1
max_temp         107.5
min_temp          97.5
dtype: float64

Number of IQR outliers:
precipitation    810
max_temp         117
min_temp          17
dtype: int64


In [769]:
# Inspect precipitation outliers.
display(
    bradentonFL[outliers["precipitation"]][
        ["year", "month", "day", "precipitation"]
    ].sort_values(
        "precipitation",
        ascending=False
    ).head(20)
)

# Inspect maximum-temperature outliers.
display(
    bradentonFL[outliers["max_temp"]][
        ["year", "month", "day", "max_temp"]
    ].sort_values(
        "max_temp",
        ascending=False
    ).head(20)
)

# Inspect minimum-temperature outliers.
display(
    bradentonFL[outliers["min_temp"]][
        ["year", "month", "day", "min_temp"]
    ].sort_values(
        "min_temp"
    ).head(20)
)

,year,month,day,precipitation
968,2017,8,26,11.65
2827,2022,9,28,7.09
3503,2024,8,4,7.00
489,2016,5,4,6.74
608,2016,8,31,6.71
3504,2024,8,5,6.30
1980,2020,6,3,6.03
2141,2020,11,11,5.80
3569,2024,10,9,4.62
1449,2018,12,20,4.30


,year,month,day,max_temp
40,2015,2,10,63.0
367,2016,1,3,63.0
708,2016,12,9,63.0
420,2016,2,25,63.0
421,2016,2,26,63.0
1110,2018,1,15,63.0
1102,2018,1,7,63.0
1176,2018,3,22,63.0
1125,2018,1,30,63.0
1096,2018,1,1,63.0


,year,month,day,min_temp
1113,2018,1,18,29.0
50,2015,2,20,34.0
2915,2022,12,25,34.0
1075,2017,12,11,34.0
2914,2022,12,24,35.0
2587,2022,1,31,35.0
1074,2017,12,10,35.0
389,2016,1,25,36.0
1100,2018,1,5,36.0
1099,2018,1,4,36.0


## florida_cyanotoxins_2015-01-01_to_2026-06-17.csv

In [770]:
print("Shape:", floridaCyanotoxins.shape)

Shape: (98884, 65)


In [771]:
floridaCyanotoxins.head()

,Org_Identifier,Org_FormalName,Project_Identifier,Location_Identifier,Location_Name,Location_Type,Location_State,Location_HUCEightDigitCode,Location_HUCTwelveDigitCode,Location_TribalLand,Location_Latitude,Location_Longitude,Activity_ActivityIdentifier,Activity_TypeCode,Activity_Media,Activity_MediaSubdivision,ActivityBiological_AssemblageSa,Activity_StartDate,Activity_StartTime,Activity_StartTimeZone,Activity_DepthHeightMeasure,Activity_DepthHeightMeasureUnit,SampleCollectionMethod_Name,Result_ResultDetectionCondition,Result_Characteristic,Result_CharacteristicUserSuppli,Result_MethodSpeciation,Result_SampleFraction,ResultBiological_Intent,ResultBiological_Taxon,ResultDepthHeight_Measure,ResultDepthHeight_MeasureUnit,Result_MeasureIdentifier,Reslt_Measre,Result_MeasureUnit,Result_MeasureQualifierCode,Result_MeasureStatusIdentifier,Result_StatisticalBase,Result_MeasureValueType,DataQuality_ResultComment,DetectionLimit_TypeA,DetectionLimit_MeasureA,DetectionLimit_MeasureUnitA,DetectionLimit_CommentA,DetectionLimit_TypeB,DetectionLimit_MeasureB,DetectionLimit_MeasureUnitB,DetectionLimit_CommentB,ResultAnalyticalMethod_Identifi,ResultAnalyticalMethod_Identi_1,ResultAnalyticalMethod_Name,LabInfo_AnalysisStartDate,LabInfo_AnalysisStartTime,LabInfo_AnalysisStartTimeZone,ResultAnalyticalMethod_Url,Result_DetectionLimitProfileDow,Result_LabSamplePrepProfileDown,ProviderName,Result_CharacteristicComparable,Result_CharacteristicGroup,Org_Type,LastChangeDate,USGSpcode,ObjectId,Value_control
0,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10246,NCA_FL-10246,Ocean,Florida,NaN,NaN,NaN,27.887829,-82.557660,NARS_WQX-NCA_FL-10246:06012015:SR:AT,Sample-Routine,Water,NaN,NaN,1.433130e+12,NaN,NaN,NaN,NaN,NCCA Grab Sample,NaN,Microcystin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STORET-937617381,NaN,ug/L,NaN,Final,NaN,Actual,NaN,Minimum Reporting Level,0.263,ug/L,NaN,NaN,NaN,NaN,NaN,520011,ABRAXIS LLC,"Microcystins and Nodularins by Immunoassay, Mi...",NaN,NaN,NaN,NaN,https://www.waterqualitydata.us/data/providers...,NaN,STORET,NaN,"Cyanotoxins, Phytotoxins",Federal/US Government,Fri Jan 31 08:59:46 GMT 2025,NaN,23431,A
1,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10245,NCA_FL-10245,Ocean,Florida,NaN,NaN,NaN,27.906991,-82.480553,NARS_WQX-NCA_FL-10245:06022015:SR:AT,Sample-Routine,Water,NaN,NaN,1.433220e+12,NaN,NaN,NaN,NaN,NCCA Grab Sample,NaN,Microcystin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STORET-937617442,NaN,ug/L,NaN,Final,NaN,Actual,NaN,Minimum Reporting Level,0.263,ug/L,NaN,NaN,NaN,NaN,NaN,520011,ABRAXIS LLC,"Microcystins and Nodularins by Immunoassay, Mi...",NaN,NaN,NaN,NaN,https://www.waterqualitydata.us/data/providers...,NaN,STORET,NaN,"Cyanotoxins, Phytotoxins",Federal/US Government,Fri Jan 31 08:59:46 GMT 2025,NaN,22625,A
2,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10297,NCA_FL-10297,Ocean,Florida,NaN,NaN,NaN,27.936200,-82.426381,NARS_WQX-NCA_FL-10297:06022015:SR:AT,Sample-Routine,Water,NaN,NaN,1.433220e+12,NaN,NaN,NaN,NaN,NCCA Grab Sample,NaN,Microcystin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STORET-937617450,NaN,ug/L,NaN,Final,NaN,Actual,NaN,Minimum Reporting Level,0.263,ug/L,NaN,NaN,NaN,NaN,NaN,520011,ABRAXIS LLC,"Microcystins and Nodularins by Immunoassay, Mi...",NaN,NaN,NaN,NaN,https://www.waterqualitydata.us/data/providers...,NaN,STORET,NaN,"Cyanotoxins, Phytotoxins",Federal/US Government,Fri Jan 31 08:59:46 GMT 2025,NaN,22713,A
3,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10299,NCA_FL-10299,Ocean,Florida,NaN,NaN,NaN,27.646342,-82.703638,NARS_WQX-NCA_FL-10299:06032015:SR:AT,Sample-Routine,Water,NaN,NaN,1.433300e+12,NaN,NaN,NaN,NaN,NCCA Grab Sample,NaN,Microcystin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STORET-937617465,NaN,ug/L,NaN,Final,NaN,Actual,NaN,Minimum Reporting Level,0.263,ug/L,NaN,NaN,NaN,NaN,NaN,520011,ABRAXIS LLC,"Microcystins and Nodularins by Immunoassay, Mi...",NaN,NaN,Na

In [772]:
# Check missing values.
print(
    floridaCyanotoxins.isna().sum()[
        floridaCyanotoxins.isna().sum() > 0
    ]
)

Location_HUCEightDigitCode           188
Location_HUCTwelveDigitCode          308
Location_TribalLand                98884
Activity_MediaSubdivision            236
ActivityBiological_AssemblageSa    98884
Activity_StartTime                   220
Activity_StartTimeZone               220
Activity_DepthHeightMeasure          228
Activity_DepthHeightMeasureUnit      228
Result_ResultDetectionCondition    98884
Result_CharacteristicUserSuppli    98884
Result_MethodSpeciation            98884
Result_SampleFraction                220
ResultBiological_Intent            98884
ResultBiological_Taxon             98884
ResultDepthHeight_Measure          98884
ResultDepthHeight_MeasureUnit      98884
Reslt_Measre                         131
Result_MeasureUnit                    60
Result_MeasureQualifierCode        98884
Result_StatisticalBase             98884
DataQuality_ResultComment           1772
DetectionLimit_TypeA                  60
DetectionLimit_MeasureA               60
DetectionLimit_M

In [773]:
# Remove columns that contain no data.
floridaCyanotoxins = floridaCyanotoxins.dropna(
    axis=1,
    how="all"
)

print("Shape after removing empty columns:", floridaCyanotoxins.shape)

Shape after removing empty columns: (98884, 45)


### Cyanotoxin Missing Values

`Reslt_Measre` is the primary measurement used for the analysis. Rows without
this measurement cannot contribute to the cyanotoxin analysis and are therefore
removed.

Other numerical fields, such as depth or geographic coordinates, are not
automatically median-imputed because replacing them with a typical value could
create artificial sampling information.

Categorical fields with missing values are represented as `"Unknown"`.

In [774]:
# Remove rows where the primary cyanotoxin measurement is missing.
floridaCyanotoxins = floridaCyanotoxins.dropna(
    subset=["Reslt_Measre"]
)

In [775]:
# Separate numerical and categorical columns for missing-value treatment
numerical_cols = floridaCyanotoxins.select_dtypes(include=["number"]).columns
categorical_cols = floridaCyanotoxins.select_dtypes(include=["object"]).columns

In [776]:
# Fill missing categorical values with "Unknown"
floridaCyanotoxins[categorical_cols] = floridaCyanotoxins[
    categorical_cols
].fillna("Unknown")

In [777]:
print("Remaining missing values:")
print(
    floridaCyanotoxins.isna().sum()[
        floridaCyanotoxins.isna().sum() > 0
    ]
)

Remaining missing values:
Location_HUCEightDigitCode      93
Location_HUCTwelveDigitCode    177
Activity_DepthHeightMeasure     97
DetectionLimit_MeasureA         60
dtype: int64


### Cyanotoxin Outlier Detection

IQR is applied to `Reslt_Measre`, the primary cyanotoxin measurement.

Because cyanotoxin concentrations can be strongly right-skewed, unusually high
measurements are expected to occur. Therefore, IQR outliers are investigated
before any decision to remove them.

In [778]:
# Examine the original distribution before removing any outliers.
floridaCyanotoxins["Reslt_Measre"].describe()

,Reslt_Measre
count,98753.000000
mean,0.449555
std,7.271026
min,0.060000
25%,0.100000
50%,0.250000
75%,0.250000
max,780.000000


In [779]:
# Calculate IQR bounds.
Q1 = floridaCyanotoxins["Reslt_Measre"].quantile(0.25)
Q3 = floridaCyanotoxins["Reslt_Measre"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.3f}")
print(f"Q3: {Q3:.3f}")
print(f"IQR: {IQR:.3f}")
print(f"Lower bound: {lower_bound:.3f}")
print(f"Upper bound: {upper_bound:.3f}")

Q1: 0.100
Q3: 0.250
IQR: 0.150
Lower bound: -0.125
Upper bound: 0.475


In [780]:
# Identify IQR outliers.
measurement_outliers = floridaCyanotoxins[
    (floridaCyanotoxins["Reslt_Measre"] < lower_bound) |
    (floridaCyanotoxins["Reslt_Measre"] > upper_bound)
]

print("Number of IQR outliers:", len(measurement_outliers))

display(
    measurement_outliers[
        [
            "Result_Characteristic",
            "Reslt_Measre",
            "Result_MeasureUnit"
        ]
    ].sort_values(
        "Reslt_Measre",
        ascending=False
    ).head(30)
)

Number of IQR outliers: 15624


,Result_Characteristic,Reslt_Measre,Result_MeasureUnit
19925,Microcystin LR,780.0,ug/L
11013,Microcystin LR,760.0,ug/L
76064,Microcystin LR,760.0,ug/L
56531,Microcystin LR,750.0,ug/L
362,Microcystin LR,730.0,ug/L
17648,Microcystin LR,600.0,ug/L
17671,Microcystin LR,600.0,ug/L
2367,Microcystin LR,490.0,ug/L
1049,Microcystin LR,450.0,ug/L
34048,Microcystin LR,410.0,ug/L


In [781]:
# Examine the largest measurements and their context.
display(
    floridaCyanotoxins.loc[
        floridaCyanotoxins["Reslt_Measre"] > 100,
        [
            "Location_Name",
            "Location_State",
            "Result_Characteristic",
            "Reslt_Measre",
            "Result_MeasureUnit",
            "Result_MeasureStatusIdentifier",
            "Result_MeasureValueType"
        ]
    ].sort_values(
        "Reslt_Measre",
        ascending=False
    )
)

,Location_Name,Location_State,Result_Characteristic,Reslt_Measre,Result_MeasureUnit,Result_MeasureStatusIdentifier,Result_MeasureValueType
19925,Lake Okeechobee - Pahokee Marina,Florida,Microcystin LR,780.0,ug/L,Final,Actual
11013,L004,Florida,Microcystin LR,760.0,ug/L,Final,Actual
76064,C44 Canal - Timer Powers Park Boat Ramp,Florida,Microcystin LR,760.0,ug/L,Final,Actual
56531,Moody Lake - SE,Florida,Microcystin LR,750.0,ug/L,Final,Actual
362,S352,Florida,Microcystin LR,730.0,ug/L,Final,Actual
17648,Harbor Isles - Southern Lobe,Florida,Microcystin LR,600.0,ug/L,Final,Actual
17671,Harbor Isles - Southern Lobe,Florida,Microcystin LR,600.0,ug/L,Final,Actual
2367,C44S80 St. Lucie Lock,Florida,Microcystin LR,490.0,ug/L,Final,Actual
1049,S79 Upstream of Lock,Florida,Microcystin LR,450.0,ug/L,Final,Actual
34048,Boat Ramp,Florida,Microcystin LR,410.0,ug/L,Final,Actual


### Cyanotoxin Outlier Assessment

The IQR method identified unusually high cyanotoxin measurements. These values
were investigated using their characteristics, measurement units, locations,
and measurement status.

The extreme measurements were retained because they represent actual
measurements with consistent units and identifiable sampling locations. High
cyanotoxin concentrations may also contain important information about periods
of elevated algal bloom activity.

habsos_cellcounts.csv

In [782]:
print("Shape:", habsos.shape)

Shape: (219152, 26)


In [783]:
habsos.head()

,OBJECTID,DESCRIPTION,LATITUDE,LONGITUDE,STATE_ID,SAMPLE_DATE,SAMPLE_DEPTH,GENUS,SPECIES,CATEGORY,CELLCOUNT,CELLCOUNT_UNIT,CELLCOUNT_QA,SALINITY,SALINITY_UNIT,SALINITY_QA,WATER_TEMP,WATER_TEMP_UNIT,WATER_TEMP_QA,WIND_DIR,WIND_DIR_UNIT,WIND_DIR_QA,WIND_SPEED,WIND_SPEED_UNIT,WIND_SPEED_QA,QA_COMMENT
0,1,Tom Adams Bridge (Lemon Bay),26.93450,-82.3535,FL,1953-08-19 00:00:00,0.5,Karenia,brevis,medium,116000.0,cells/L,1,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day
1,2,Naples Pier,26.13163,-81.8063,FL,1953-08-26 00:00:00,0.5,Karenia,brevis,not observed,0.0,cells/L,1,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day
2,3,Piney Point; Lee County,26.53340,-81.9796,FL,1953-08-26 00:00:00,0.5,Karenia,brevis,not observed,0.0,cells/L,1,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day
3,4,Cortez Bridge,27.46840,-82.6937,FL,1953-08-26 00:00:00,0.5,Karenia,brevis,medium,178000.0,cells/L,1,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day
4,5,Sarasota Bay; 3 mi North of bridge,27.37750,-82.5708,FL,1953-09-03 00:00:00,0.5,Karenia,brevis,high,1732000.0,cells/L,1,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day


In [784]:
# Remove completely empty columns.
habsos = habsos.dropna(
    axis=1,
    how="all"
)

# WIND_SPEED and WIND_SPEED_UNIT do not contain usable information.
habsos = habsos.drop(
    columns=["WIND_SPEED", "WIND_SPEED_UNIT"]
)

print("Shape after column cleaning:", habsos.shape)

Shape after column cleaning: (219152, 22)


### HABSOS Missing Values

Water temperature and salinity are sequential environmental measurements.
Short gaps can therefore be estimated from surrounding observations using linear
interpolation.

A limit of two observations is used so that only short gaps are filled.

Categorical variables are filled with `"Unknown"`.

If both salinity and water temperature are missing for an observation, the
observation cannot contribute useful environmental measurements and is removed.

Sample depth is not a sequential measurement in this context, so missing depth
values are represented using the median depth rather than interpolation.

In [785]:
# Linearly interpolate only short gaps in environmental measurements.
habsos["WATER_TEMP"] = habsos[
    "WATER_TEMP"
].interpolate(
    method="linear",
    limit=2
)

habsos["SALINITY"] = habsos[
    "SALINITY"
].interpolate(
    method="linear",
    limit=2
)

In [786]:
# Fill missing categorical metadata with "Unknown".
categorical_cols = habsos.select_dtypes(
    include=["object"]
).columns

habsos[categorical_cols] = habsos[
    categorical_cols
].fillna("Unknown")

In [787]:
# Remove observations missing both major environmental measurements.
habsos = habsos.dropna(
    subset=["SALINITY", "WATER_TEMP"],
    how="all"
)

In [788]:
# Fill missing sample depth using the median.
habsos["SAMPLE_DEPTH"] = habsos[
    "SAMPLE_DEPTH"
].fillna(
    habsos["SAMPLE_DEPTH"].median()
)

In [789]:
print("Remaining missing values:")
print(
    habsos.isna().sum()[
        habsos.isna().sum() > 0
    ]
)

Remaining missing values:
SALINITY       9464
WATER_TEMP    11093
dtype: int64


### HABSOS Outlier Detection

IQR is applied separately to salinity, water temperature, and cell counts.

Cell counts are strongly right-skewed, so a log transformation is used before
IQR screening. The original `CELLCOUNT` values are retained.

In [790]:
# -----------------------------
# Salinity
# -----------------------------

Q1 = habsos["SALINITY"].quantile(0.25)
Q3 = habsos["SALINITY"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Salinity")
print(f"Q1: {Q1:.3f}")
print(f"Q3: {Q3:.3f}")
print(f"IQR: {IQR:.3f}")
print(f"Lower bound: {lower_bound:.3f}")
print(f"Upper bound: {upper_bound:.3f}")

salinity_outliers = habsos[
    (habsos["SALINITY"] < lower_bound) |
    (habsos["SALINITY"] > upper_bound)
]

print("Number of IQR outliers:", len(salinity_outliers))

display(
    salinity_outliers[
        [
            "OBJECTID",
            "SALINITY",
            "SALINITY_UNIT",
            "SALINITY_QA",
            "QA_COMMENT"
        ]
    ].sort_values(
        "SALINITY",
        ascending=False
    ).head(20)
)

# Remove values above the selected physically plausible salinity range.
habsos = habsos.drop(
    habsos[
        habsos["SALINITY"] > 50
    ].index
)

print("Maximum salinity after cleaning:")
print(habsos["SALINITY"].max())

Salinity
Q1: 28.190
Q3: 35.000
IQR: 6.810
Lower bound: 17.975
Upper bound: 45.215
Number of IQR outliers: 7053


,OBJECTID,SALINITY,SALINITY_UNIT,SALINITY_QA,QA_COMMENT
204621,597495,308.00,ppt,4,Unknown
183985,406328,222.70,ppt,4,WDIR unexpected data in input;CATEGORIES from ...
204619,597493,161.00,Unknown,9,Unknown
204618,597492,87.50,Unknown,9,Unknown
196276,580077,72.00,ppt,4,WTEMP inferred as Fahrenheit based on value an...
196277,580078,64.80,Unknown,9,Unknown
182167,346512,57.80,ppt,4,SALINITY value (57.80) exceeds valid range (0....
196278,580079,57.60,Unknown,9,Unknown
4921,4922,57.00,ppt,4,SALINITY value (57.00) exceeds valid range (0....
5025,5026,56.00,ppt,4,SALINITY value (56.00) exceeds valid range (0....


Maximum salinity after cleaning:
50.0


In [791]:
# -----------------------------
# Water temperature
# -----------------------------

Q1 = habsos["WATER_TEMP"].quantile(0.25)
Q3 = habsos["WATER_TEMP"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Water temperature")
print(f"Q1: {Q1:.3f}")
print(f"Q3: {Q3:.3f}")
print(f"IQR: {IQR:.3f}")
print(f"Lower bound: {lower_bound:.3f}")
print(f"Upper bound: {upper_bound:.3f}")

water_temp_outliers = habsos[
    (habsos["WATER_TEMP"] < lower_bound) |
    (habsos["WATER_TEMP"] > upper_bound)
]

print("Number of IQR outliers:", len(water_temp_outliers))

display(
    water_temp_outliers[
        [
            "OBJECTID",
            "WATER_TEMP",
            "WATER_TEMP_UNIT",
            "WATER_TEMP_QA",
            "QA_COMMENT"
        ]
    ].sort_values(
        "WATER_TEMP",
        ascending=False
    ).head(20)
)

Water temperature
Q1: 20.980
Q3: 29.000
IQR: 8.020
Lower bound: 8.950
Upper bound: 41.030
Number of IQR outliers: 921


,OBJECTID,WATER_TEMP,WATER_TEMP_UNIT,WATER_TEMP_QA,QA_COMMENT
184352,406695,1503.888889,deg. C,4,WTEMP value (1503.89) exceeds valid range (-2....
184302,406645,568.333333,deg. C,4,WTEMP value (568.33) exceeds valid range (-2.0...
213213,611134,149.111111,deg. C,4,Unknown
186448,434867,100.000000,deg. C,4,WTEMP value (100.00) exceeds valid range (-2.0...
212474,609991,8.940000,deg. C,1,Unknown
216736,616391,8.900000,deg. C,1,Unknown
217299,616954,8.900000,deg. C,1,Unknown
217300,616955,8.900000,deg. C,1,Unknown
93257,93613,8.900000,deg. C,1,Unknown
190310,547984,8.900000,deg. C,1,Unknown


In [792]:
# Remove water temperatures outside the selected physically plausible range.
habsos = habsos.drop(
    habsos[
        (habsos["WATER_TEMP"] > 40) |
        (habsos["WATER_TEMP"] < -2)
    ].index
)

print("Water temperature range after cleaning:")
print(habsos["WATER_TEMP"].describe())

Water temperature range after cleaning:
count    135843.000000
mean         24.519477
std           5.324568
min           0.000000
25%          20.980000
50%          25.200000
75%          29.000000
max          33.000000
Name: WATER_TEMP, dtype: float64


In [793]:
# -----------------------------
# Cell count
# -----------------------------

# Log-transform cell counts to reduce strong right-skew.
habsos["LOG_CELLCOUNT"] = np.log1p(
    habsos["CELLCOUNT"]
)

Q1 = habsos["LOG_CELLCOUNT"].quantile(0.25)
Q3 = habsos["LOG_CELLCOUNT"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Log cell count")
print(f"Q1: {Q1:.3f}")
print(f"Q3: {Q3:.3f}")
print(f"IQR: {IQR:.3f}")
print(f"Lower bound: {lower_bound:.3f}")
print(f"Upper bound: {upper_bound:.3f}")

cellcount_outliers = habsos[
    (habsos["LOG_CELLCOUNT"] < lower_bound) |
    (habsos["LOG_CELLCOUNT"] > upper_bound)
]

print("Number of IQR outliers:", len(cellcount_outliers))

display(
    cellcount_outliers[
        [
            "OBJECTID",
            "CELLCOUNT",
            "LOG_CELLCOUNT",
            "CELLCOUNT_QA",
            "QA_COMMENT"
        ]
    ].sort_values(
        "CELLCOUNT",
        ascending=False
    ).head(30)
)

Log cell count
Q1: 0.000
Q3: 0.000
IQR: 0.000
Lower bound: 0.000
Upper bound: 0.000
Number of IQR outliers: 29450


,OBJECTID,CELLCOUNT,LOG_CELLCOUNT,CELLCOUNT_QA,QA_COMMENT
35164,35265,358000000.0,19.696044,1,Unknown
175953,181053,189456000.0,19.059667,1,CATEGORIES from provider;WTEMP inferred as Fah...
161844,164254,186266667.0,19.042690,1,Unknown
66166,66279,162188000.0,18.904267,1,Unknown
64846,64955,152210000.0,18.840772,1,Unknown
63895,63998,151100000.0,18.833452,1,Unknown
175941,181040,138000000.0,18.742764,1,CATEGORIES from provider;WTEMP inferred as Fah...
38088,38189,116016000.0,18.569239,1,Unknown
37947,38048,108672000.0,18.503845,1,Unknown
20859,20906,107000000.0,18.488339,1,Unknown


### HABSOS Outlier Assessment

Salinity values above 50 and water temperatures above 40°C or below -2°C were
removed because they fall outside the selected physically plausible ranges.

Other IQR outliers were retained when they could represent legitimate
environmental variation. In particular, high cell counts may represent periods
of increased algal abundance rather than erroneous measurements.

MoteMarine_BottomTemp.csv

In [794]:
print("Shape:", motemarine.shape)

Shape: (31730, 6)


In [795]:
motemarine.head()

,time,latitude,longitude,z,sea_water_temperature,sea_water_temperature_qc_agg
0,UTC,degrees_north,degrees_east,m,degree_Celsius,NaN
1,2020-08-13T15:00:00Z,27.3292,-82.5564,-2.0,30.929,1.0
2,2020-08-13T16:00:00Z,27.3292,-82.5564,-2.0,31.073,1.0
3,2020-08-13T17:00:00Z,27.3292,-82.5564,-2.0,31.217,1.0
4,2020-08-13T18:00:00Z,27.3292,-82.5564,-2.0,31.274,1.0


In [796]:
# Check missing values.
print(
    motemarine.isna().sum()[
        motemarine.isna().sum() > 0
    ]
)

sea_water_temperature_qc_agg    1
dtype: int64


In [797]:
# Convert environmental measurements to numeric.
measurement_cols = [
    "latitude",
    "longitude",
    "z",
    "sea_water_temperature"
]

motemarine[measurement_cols] = motemarine[
    measurement_cols
].apply(
    pd.to_numeric,
    errors="coerce"
)

### Mote Marine Missing Values

The missing QC unit is metadata and cannot be reliably inferred, so it is left
missing.

Environmental measurement values are not median-imputed because there is not
enough justification to replace an unrecorded observation with a typical value.

In [798]:
print("Remaining missing values:")
print(
    motemarine.isna().sum()[
        motemarine.isna().sum() > 0
    ]
)

Remaining missing values:
latitude                        1
longitude                       1
z                               1
sea_water_temperature           1
sea_water_temperature_qc_agg    1
dtype: int64


### Mote Marine Outlier Detection

IQR is used to screen the bottom-water temperature measurement.

Latitude and longitude are not treated as generic outliers because an extreme
coordinate may simply represent a legitimate sampling location.

In [799]:
# Use bottom-water temperature as the environmental measurement for
# outlier screening.
values = motemarine["sea_water_temperature"].dropna()

Q1 = values.quantile(0.25)
Q3 = values.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.3f}")
print(f"Q3: {Q3:.3f}")
print(f"IQR: {IQR:.3f}")
print(f"Lower bound: {lower_bound:.3f}")
print(f"Upper bound: {upper_bound:.3f}")

temperature_outliers = motemarine[
    (motemarine["sea_water_temperature"] < lower_bound) |
    (motemarine["sea_water_temperature"] > upper_bound)
]

print("Number of IQR outliers:", len(temperature_outliers))

display(temperature_outliers.head(20))

Q1: 20.630
Q3: 29.439
IQR: 8.809
Lower bound: 7.416
Upper bound: 42.653
Number of IQR outliers: 0


,time,latitude,longitude,z,sea_water_temperature,sea_water_temperature_qc_agg


NutrientsFL_2006_2025.csv

In [800]:
print("Shape:", nutrientsFL.shape)

Shape: (514087, 18)


In [801]:
nutrientsFL.head()

,DataSource,StationID,Actual_DataSource,Actual_StationID,Activity_Start_Date,Activity_Start_Time,Activity_Type,RelativeDepth,Activity_Depth,Activity_Depth_Unit,Characteristic,Result_Value,Result_Unit,Value_Qualifier,Sample_Fraction,MDL,MDL_Unit,Result_Comment
0,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,"BOD, Biochemical oxygen demand",2.16,mg/l,I,NaN,NaN,NaN,NaN
1,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,"Chlorophyll a, corrected for pheophytin",21.40,ug/l,NaN,NaN,NaN,NaN,NaN
2,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,Dissolved oxygen (DO),4.25,mg/l,NaN,NaN,NaN,NaN,NaN
3,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,Dissolved oxygen saturation,54.90,percent (%),NaN,NaN,NaN,NaN,NaN
4,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,Fecal Coliform,330.00,cfu/100ml,NaN,NaN,NaN,NaN,NaN


In [802]:
missing_values = nutrientsFL.isna().sum()

print(
    missing_values[
        missing_values > 0
    ]
)

Activity_Start_Time     82051
Activity_Type           20334
RelativeDepth           50644
Activity_Depth           1911
Activity_Depth_Unit      2106
Result_Unit             54597
Value_Qualifier        426211
Sample_Fraction        323680
MDL                    216198
MDL_Unit               241369
Result_Comment         426364
dtype: int64


In [803]:
# Calculate missing-value percentages.
null_percent = nutrientsFL.isna().mean() * 100

print(
    null_percent[
        null_percent > 0
    ].sort_values(ascending=False)
)

Result_Comment         82.936157
Value_Qualifier        82.906395
Sample_Fraction        62.962106
MDL_Unit               46.951002
MDL                    42.054749
Activity_Start_Time    15.960528
Result_Unit            10.620187
RelativeDepth           9.851251
Activity_Type           3.955362
Activity_Depth_Unit     0.409658
Activity_Depth          0.371727
dtype: float64


### NutrientsFL Missing Values

The NutrientsFL dataset contains different types of environmental measurements
and sampling metadata.

Because these variables have different meanings, missing values are handled
according to the specific field:

- Categorical metadata → `"Unknown"` when appropriate.
- Primary measurement values → left missing unless the observation is unusable.
- Detection limits → left missing because they are measurement metadata.
- Sampling dates/times → left missing because they cannot be reliably inferred.
- Sampling depth → left missing when it was not recorded.
- Measurement units → left missing when the missing value has a meaningful
  relationship to the measurement, such as dimensionless pH.

In [804]:
# Categorical fields where "Unknown" is an appropriate representation
# of an unreported value.
categorical_fill_cols = [
    "Activity_Type",
    "RelativeDepth",
    "Sample_Fraction",
    "MDL_Unit",
    "Value_Qualifier"
]

nutrientsFL[categorical_fill_cols] = nutrientsFL[
    categorical_fill_cols
].fillna("Unknown")

print("Remaining missing values:")
print(
    nutrientsFL.isna().sum()[
        nutrientsFL.isna().sum() > 0
    ]
)

Remaining missing values:
Activity_Start_Time     82051
Activity_Depth           1911
Activity_Depth_Unit      2106
Result_Unit             54597
MDL                    216198
Result_Comment         426364
dtype: int64


### NutrientsFL Outlier Detection

`Result_Value` contains many different environmental measurements. Applying one
IQR calculation to the entire column would be inappropriate because the values
have different units and distributions.

Instead, IQR is calculated separately for each `Characteristic`.

In [805]:
# Get all unique characteristics that have numerical Result_Value data.
characteristics = nutrientsFL[
    "Characteristic"
].dropna().unique()

print("Number of characteristics:", len(characteristics))
print(characteristics)

Number of characteristics: 21
['BOD, Biochemical oxygen demand'
 'Chlorophyll a, corrected for pheophytin' 'Dissolved oxygen (DO)'
 'Dissolved oxygen saturation' 'Fecal Coliform' 'Nitrogen'
 'Nitrogen, ammonia as N' 'Nitrogen, Kjeldahl'
 'Nitrogen, Nitrite (NO2) + Nitrate (NO3) as N' 'pH' 'Phosphorus as P'
 'Phosphorus, phosphate (PO4) as P' 'Specific conductance'
 'Temperature, water' 'Total Suspended Solids (TSS)' 'Turbidity'
 'True Color' 'Chlorophyll a, uncorrected for pheophytin'
 'Temperature, Water' 'Dissolved Oxygen Saturation' 'Specific Conductance']


In [806]:
# Apply IQR outlier detection separately to each characteristic.
#
# Result_Value contains measurements with different units and distributions,
# so one overall IQR calculation would not be appropriate. Instead, each
# characteristic gets its own Q1, Q3, IQR, and outlier boundaries.

for characteristic in characteristics:

    # Get the Result_Value measurements for this characteristic.
    values = nutrientsFL.loc[
        nutrientsFL["Characteristic"] == characteristic,
        "Result_Value"
    ].dropna()

    # Skip characteristics that do not have numerical measurements.
    if len(values) == 0:
        continue

    # Calculate the first and third quartiles.
    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)

    # Calculate the interquartile range.
    IQR = Q3 - Q1

    # Calculate the IQR outlier boundaries.
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify measurements outside the IQR boundaries.
    outliers = nutrientsFL.loc[
        (nutrientsFL["Characteristic"] == characteristic) &
        (
            (nutrientsFL["Result_Value"] < lower_bound) |
            (nutrientsFL["Result_Value"] > upper_bound)
        )
    ]

    print("\n" + "=" * 70)
    print(characteristic)
    print("=" * 70)

    print(f"Number of observations: {len(values)}")
    print(f"Q1: {Q1:.3f}")
    print(f"Q3: {Q3:.3f}")
    print(f"IQR: {IQR:.3f}")
    print(f"Lower bound: {lower_bound:.3f}")
    print(f"Upper bound: {upper_bound:.3f}")
    print(f"Number of IQR outliers: {len(outliers)}")

    # Display the largest outliers for inspection.
    if len(outliers) > 0:
        display(
            outliers[
                [
                    "Activity_Start_Date",
                    "StationID",
                    "Result_Value",
                    "Result_Unit",
                    "Value_Qualifier",
                    "Result_Comment"
                ]
            ].sort_values(
                "Result_Value",
                ascending=False
            ).head(20)
        )


BOD, Biochemical oxygen demand
Number of observations: 25640
Q1: 0.810
Q3: 1.710
IQR: 0.900
Lower bound: -0.540
Upper bound: 3.060
Number of IQR outliers: 2255


,Activity_Start_Date,StationID,Result_Value,Result_Unit,Value_Qualifier,Result_Comment
143768,4/6/2009 0:00,MY-A,20.0,mg/l,I,NaN
1113,6/22/2011 0:00,ALL,20.0,mg/l,Unknown,NaN
1114,6/22/2011 0:00,ALL,20.0,mg/l,Unknown,NaN
9784,5/20/2015 0:00,ALL-3,19.6,mg/l,Unknown,NaN
9785,5/20/2015 0:00,ALL-3,19.6,mg/l,Unknown,NaN
98299,3/21/2011 0:00,GOT-D,19.6,mg/l,Unknown,NaN
98300,3/21/2011 0:00,GOT-D,19.6,mg/l,Unknown,NaN
5450,6/22/2011 0:00,ALL-2,17.3,mg/l,Unknown,NaN
5451,6/22/2011 0:00,ALL-2,17.3,mg/l,Unknown,NaN
32987,5/11/2020 14:14,C341-17th,17.3,mg/l,Unknown,NaN



Chlorophyll a, corrected for pheophytin
Number of observations: 26377
Q1: 2.440
Q3: 10.500
IQR: 8.060
Lower bound: -9.650
Upper bound: 22.590
Number of IQR outliers: 2556


,Activity_Start_Date,StationID,Result_Value,Result_Unit,Value_Qualifier,Result_Comment
198501,2/15/2018 0:00,SW-Head,4333.0,ug/l,Unknown,NaN
204497,6/17/2021 9:22,VG-1,2651.0,ug/l,Unknown,NaN
198541,4/17/2018 0:00,SW-Head,2403.0,ug/l,Unknown,NaN
142639,5/9/2022 9:27,ML,1573.0,ug/l,Unknown,NaN
31468,7/23/2022 12:02,BW-Tail,793.0,ug/l,Unknown,NaN
206636,6/17/2021 9:54,VG-4,674.0,ug/l,Unknown,NaN
204482,5/20/2021 9:35,VG-1,667.0,ug/l,Unknown,NaN
204633,5/23/2022 9:10,VG-1,611.0,ug/l,Unknown,NaN
206762,5/23/2022 9:26,VG-4,530.0,ug/l,Unknown,NaN
204338,7/16/2020 9:26,VG-1,518.0,ug/l,Unknown,NaN



Dissolved oxygen (DO)
Number of observations: 27143
Q1: 4.787
Q3: 7.200
IQR: 2.413
Lower bound: 1.167
Upper bound: 10.820
Number of IQR outliers: 1364


...

,Activity_Start_Date,StationID,Result_Value,Result_Unit,Value_Qualifier,Result_Comment
198511,2/15/2018 12:39,SW-Head,4333.0,ug/l,,NaN
198550,4/17/2018 13:06,SW-Head,2403.0,ug/l,,NaN
2997,2/22/2018 12:01,ALL,517.0,ug/l,,NaN
187009,5/15/2018 9:28,RBW-MIR,479.0,ug/l,,NaN
141749,12/12/2016 10:03,ML,332.0,ug/l,,NaN
203606,7/21/2016 9:30,VG-1,285.0,ug/l,,NaN
202958,10/18/2018 9:15,VEN_GAR-4,281.0,ug/l,,NaN
141570,4/18/2016 9:26,ML,269.0,ug/l,,NaN
204109,8/15/2018 9:00,VG-1,266.0,ug/l,,NaN
26792,7/12/2018 11:17,BW-Head,258.0,ug/l,,NaN



Temperature, Water
Number of observations: 11677
Q1: 70.700
Q3: 85.820
IQR: 15.120
Lower bound: 48.020
Upper bound: 108.500
Number of IQR outliers: 0

Dissolved Oxygen Saturation
Number of observations: 11759
Q1: 88.000
Q3: 104.000
IQR: 16.000
Lower bound: 64.000
Upper bound: 128.000
Number of IQR outliers: 665


,Activity_Start_Date,StationID,Result_Value,Result_Unit,Value_Qualifier,Result_Comment
313633,1/6/2021 14:35,13-1,208.0,percent (%),Unknown,NaN
237227,8/8/2018 13:03,10-1,208.0,percent (%),Unknown,NaN
237217,8/8/2018 13:02,10-1,207.0,percent (%),Unknown,NaN
485987,7/10/2019 12:43,LB-5,202.0,percent (%),Unknown,NaN
321140,8/12/2020 14:07,13-2,184.0,percent (%),Unknown,NaN
512159,8/8/2018 12:46,US-5,179.0,percent (%),Unknown,NaN
306362,7/6/2022 13:57,11-5,179.0,percent (%),Unknown,NaN
512169,8/8/2018 12:47,US-5,178.0,percent (%),Unknown,NaN
261725,5/7/2025 14:03,10-4,174.0,percent (%),Unknown,NaN
261736,5/7/2025 14:04,10-4,172.0,percent (%),Unknown,NaN



Specific Conductance
Number of observations: 8638
Q1: 48300.000
Q3: 53500.000
IQR: 5200.000
Lower bound: 40500.000
Upper bound: 61300.000
Number of IQR outliers: 499


,Activity_Start_Date,StationID,Result_Value,Result_Unit,Value_Qualifier,Result_Comment
321424,9/1/2021 13:59,13-2,40400.0,umho,Unknown,NaN
321418,9/1/2021 13:58,13-2,40400.0,umho,Unknown,NaN
486550,9/1/2021 10:01,LB-5,40400.0,umho,Unknown,NaN
306911,8/7/2024 14:07,11-5,40400.0,umho,Unknown,NaN
486544,9/1/2021 10:00,LB-5,40400.0,umho,Unknown,NaN
443079,10/12/2022 11:38,DR-4,40400.0,umho,Unknown,NaN
436701,9/9/2020 12:14,DR-3,40400.0,umho,Unknown,NaN
404084,9/9/2020 10:26,16-3,40400.0,umho,Unknown,NaN
366716,8/11/2021 12:20,14-3,40400.0,umho,Unknown,NaN
486554,9/1/2021 10:02,LB-5,40400.0,umho,Unknown,NaN


In [807]:
# Remove clearly erroneous pH measurements.
nutrientsFL = nutrientsFL.drop(
    nutrientsFL[
        (nutrientsFL["Characteristic"] == "pH") &
        (nutrientsFL["Result_Value"] >= 22)
    ].index
)

# Remove clearly implausible water-temperature measurements.
nutrientsFL = nutrientsFL.drop(
    nutrientsFL[
        (nutrientsFL["Characteristic"] == "Temperature, water") &
        (nutrientsFL["Result_Value"] > 110)
    ].index
)

# Remove clearly implausible dissolved oxygen measurements.
nutrientsFL = nutrientsFL.drop(
    nutrientsFL[
        (nutrientsFL["Characteristic"] == "Dissolved oxygen (DO)") &
        (nutrientsFL["Result_Value"] > 20)
    ].index
)